In [1]:
!nvidia-smi

Sun May  3 06:36:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile reduction.cu
#include <iostream>
using namespace std;

__global__ void reduce(int *arr, int *sum, int *minVal, int *maxVal, int n) {
    int tid = threadIdx.x;

    if (tid == 0) {
        int s = 0;
        int mn = arr[0];
        int mx = arr[0];

        for (int i = 0; i < n; i++) {
            s += arr[i];

            if (arr[i] < mn) mn = arr[i];
            if (arr[i] > mx) mx = arr[i];
        }

        *sum = s;
        *minVal = mn;
        *maxVal = mx;
    }
}

int main() {
    int n;

    cout << "Enter number of elements: ";
    cin >> n;

    int *arr = new int[n];

    cout << "Enter elements:\n";
    for (int i = 0; i < n; i++) {
        cin >> arr[i];
    }

    int *d_arr, *d_sum, *d_min, *d_max;
    int h_sum, h_min, h_max;

    // Allocate memory on GPU
    cudaMalloc(&d_arr, n * sizeof(int));
    cudaMalloc(&d_sum, sizeof(int));
    cudaMalloc(&d_min, sizeof(int));
    cudaMalloc(&d_max, sizeof(int));

    // Copy data to GPU
    cudaMemcpy(d_arr, arr, n * sizeof(int), cudaMemcpyHostToDevice);

    // Launch kernel
    reduce<<<1, 1>>>(d_arr, d_sum, d_min, d_max, n);

    // Copy results back to CPU
    cudaMemcpy(&h_sum, d_sum, sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(&h_min, d_min, sizeof(int), cudaMemcpyDeviceToHost);
    cudaMemcpy(&h_max, d_max, sizeof(int), cudaMemcpyDeviceToHost);

    float avg = (float)h_sum / n;

    cout << "\nResults:\n";
    cout << "Sum = " << h_sum << endl;
    cout << "Min = " << h_min << endl;
    cout << "Max = " << h_max << endl;
    cout << "Average = " << avg << endl;

    // Free memory
    cudaFree(d_arr);
    cudaFree(d_sum);
    cudaFree(d_min);
    cudaFree(d_max);
    delete[] arr;

    return 0;
}

Writing reduction.cu


In [3]:
!nvcc reduction.cu -o reduction

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./reduction

Enter number of elements: 5
Enter elements:
56 1 34 24 6

Results:
Sum = 121
Min = 1
Max = 56
Average = 24.2
